# PN10 factor-sphere prime recovery

This executed companion reviews the frozen PN10 artifacts. The complete deterministic calculation lives in
`pn10_factor_sphere_prime_recovery.py`; an independent implementation lives in
`pn10_validate_factor_sphere.py`.

For integer `n` and factor candidate `d`,

\[
x_n(d)=\frac{2\log d}{\log n},\qquad x_n(d)+x_n(n/d)=2.
\]

The endpoints are `1` and `n`; the `1.0` ridge is `sqrt(n)`. A divisor collision at or before the ridge means
composite. A quiet walk through the ridge means prime.


In [1]:
from pathlib import Path
import csv, hashlib, json, math

HERE = Path(r'F:\SystemFormulaFolder\GIT\ARA-GIT\analysis\primes')
def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest().upper()

freeze = json.loads((HERE / 'PN10_FREEZE_MANIFEST.json').read_text(encoding='utf-8'))
result = json.loads((HERE / 'PN10_FACTOR_SPHERE_RESULTS.json').read_text(encoding='utf-8'))
print('Protocol hash:', sha256(HERE / 'PN10_FACTOR_SPHERE_PRIME_RECOVERY_PROTOCOL.md'))
print('Frozen hash:  ', freeze['protocol_sha256'])
print('Evidence:', result['evidence_class'])
print('Protected:', result['protected_material'])


Protocol hash: A46FC79D82034CB827F907C531C4DF208B9E33E3AFCE8E9E00D60E637D8F4BEE
Frozen hash:   A46FC79D82034CB827F907C531C4DF208B9E33E3AFCE8E9E00D60E637D8F4BEE
Evidence: registered exact crosswalk plus fresh cross-scale computational transfer
Protected: {'p31_primorial_wheel_constructed': False, 'r12_opened': False}


In [2]:
def x(n, d):
    return 2 * math.log(d) / math.log(n)

for n in [77, 79, 121]:
    factors = [d for d in range(1, n + 1) if n % d == 0]
    print(n, [(d, round(x(n, d), 6)) for d in factors])
print('77 factor-pair closure:', x(77, 7) + x(77, 11))
print('121 square ridge:', x(121, 11))


77 [(1, 0.0), (7, 0.895947), (11, 1.104053), (77, 2.0)]
79 [(1, 0.0), (79, 2.0)]
121 [(1, 0.0), (11, 1.0), (121, 2.0)]
77 factor-pair closure: 2.0
121 square ridge: 1.0


In [3]:
print('Development primes:', result['intervals']['development']['primes'])
print('Fresh evaluation primes:', result['intervals']['evaluation']['primes'])
print('First 25 fresh primes:')
print(result['exact_recovery']['first_25_evaluation_primes'])
print('\nRegistered criteria')
for name, record in result['criteria'].items():
    state = record.get('pass', record.get('all_primary_cutoffs_retain_composites'))
    print(name, 'PASS' if state else 'FAIL')


Development primes: 70435
Fresh evaluation primes: 46903
First 25 fresh primes:
[2000000011, 2000000033, 2000000063, 2000000087, 2000000089, 2000000099, 2000000137, 2000000141, 2000000143, 2000000153, 2000000203, 2000000227, 2000000239, 2000000243, 2000000269, 2000000273, 2000000279, 2000000293, 2000000323, 2000000333, 2000000357, 2000000381, 2000000393, 2000000407, 2000000413]

Registered criteria
P1_exact_prime_recovery PASS
P2_reversible_factor_pair_closure PASS
P3_prime_square_ridge PASS
P4_accumulating_information PASS
P5_cross_scale_brier PASS
P6_cross_scale_calibration PASS
L1_early_exactness_limit PASS


In [4]:
rows = list(csv.DictReader((HERE / 'PN10_FACTOR_SPHERE_TRANSFER.csv').open(encoding='utf-8')))
print('cutoff | method     | development purity | evaluation purity | transfer error | Brier | composites left')
for row in rows:
    print(f"{float(row['cutoff']):>6.2f} | {row['method']:<10} | {float(row['development_purity']):>18.6f} | {float(row['evaluation_purity']):>17.6f} | {float(row['purity_transfer_error']):>14.6f} | {float(row['evaluation_brier']):.6f} | {int(row['evaluation_remaining_composites']):>15,}")


cutoff | method     | development purity | evaluation purity | transfer error | Brier | composites left
  0.25 | ARA scaled |           0.264130 |          0.244528 |       0.019601 | 0.035508 |         144,907
  0.25 | fixed Q    |           0.264130 |          0.175887 |       0.088243 | 0.040730 |         219,763
  0.50 | ARA scaled |           0.462393 |          0.453634 |       0.008760 | 0.025634 |          56,491
  0.50 | fixed Q    |           0.460820 |          0.306842 |       0.153978 | 0.036135 |         105,954
  0.75 | ARA scaled |           0.671078 |          0.664678 |       0.006400 | 0.015731 |          23,662
  0.75 | fixed Q    |           0.673137 |          0.451481 |       0.221656 | 0.030831 |          56,984
  0.90 | ARA scaled |           0.837346 |          0.835286 |       0.002059 | 0.007726 |           9,249
  0.90 | fixed Q    |           0.831396 |          0.534045 |       0.297351 | 0.029620 |          40,923


In [5]:
validation = json.loads((HERE / 'PN10_FACTOR_SPHERE_VALIDATION.json').read_text(encoding='utf-8'))
print('Independent validation:', validation['status'])
print('Checks:', validation['passed_checks'], '/', validation['total_checks'])
print('Static figure:', HERE / 'PN10_FACTOR_SPHERE_FIGURE.png')
print('To reconstruct everything, run:')
print('  python pn10_factor_sphere_prime_recovery.py')
print('  python pn10_validate_factor_sphere.py')


Independent validation: PASS
Checks: 64 / 64
Static figure: F:\SystemFormulaFolder\GIT\ARA-GIT\analysis\primes\PN10_FACTOR_SPHERE_FIGURE.png
To reconstruct everything, run:
  python pn10_factor_sphere_prime_recovery.py
  python pn10_validate_factor_sphere.py


## Reading the result

The full ridge walk is an exact prime test because it is the classical `sqrt(n)` completeness condition in a
reversible ARA coordinate. The partial walk is not exact: at `c=0.90`, 9,249 composites remain among 56,152 fresh
survivors. Its strong result is cross-scale calibration: survivor purity transfers from the development interval to
the much larger fresh interval far better than a fixed absolute divisor cutoff.

That supports the factor-sphere coordinate as a useful representation. It does not establish a faster prime
algorithm or distinguish ARA from established relative-logarithmic factor theory by itself.
